# Project 3: Customer Behavior EDA


# Exploratory Data Analysis (EDA)
This notebook follows a reusable EDA workflow: inspect the data, check data quality, summarize distributions, visualize patterns, investigate anomalies, and document findings.

> The dataset is generated inside this notebook so the project runs without downloading files. Replace the sample data with a real CSV later if desired.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme()

## 2. Create a sample customer dataset

In [ ]:
rng = np.random.default_rng(29)
n = 400
age = rng.normal(36, 11, n).clip(18, 75).round()
income = rng.normal(55000, 16000, n).clip(15000, 130000).round(0)
visits = rng.poisson(5, n) + 1
spend = (income * 0.0007 + visits * 18 + rng.normal(0, 35, n)).clip(10, None).round(2)
df = pd.DataFrame({
    "customer_id": [f"C{i:04d}" for i in range(1, n+1)],
    "age": age.astype(int),
    "annual_income": income.astype(int),
    "city": rng.choice(["Hyderabad", "Bengaluru", "Chennai", "Pune", "Delhi"], n, p=[.28,.24,.16,.16,.16]),
    "membership": rng.choice(["Basic", "Silver", "Gold"], n, p=[.5,.3,.2]),
    "website_visits_month": visits,
    "orders_last_90_days": rng.poisson(2.5, n),
    "total_spend_90_days": spend,
    "last_purchase_date": pd.to_datetime("2025-12-31") - pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "feedback": rng.choice(["fast delivery", "good quality", "price high", "easy checkout", ""], n)
})
# Add quality issues and a few extreme values to investigate
df.loc[[5, 17, 90], "annual_income"] = np.nan
df.loc[[12, 45], "membership"] = np.nan
df.loc[25, "age"] = 150
df.loc[72, "total_spend_90_days"] = 2500
df = pd.concat([df, df.iloc[[3]]], ignore_index=True)
df.head()

## 3. Inspect the dataset

In [ ]:
print("Shape:", df.shape)
display(df.head())
df.info()
display(df.describe(include="all").T)

## 4. Data quality and cleaning

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean()*100).round(2),
    "unique_values": df.nunique(dropna=True)
})
display(quality)
eda = df.drop_duplicates().copy()
eda["annual_income"] = eda["annual_income"].fillna(eda["annual_income"].median())
eda["membership"] = eda["membership"].fillna("Unknown")
eda["feedback"] = eda["feedback"].replace("", "No feedback").fillna("No feedback")
# Flag invalid ages instead of silently changing them
eda["age_valid"] = eda["age"].between(18, 100)
display(eda.loc[~eda["age_valid"], ["customer_id", "age"]])
print("Remaining missing values:", int(eda.isna().sum().sum()))

## 5. Numerical descriptive statistics

In [ ]:
numeric = ["age", "annual_income", "website_visits_month", "orders_last_90_days", "total_spend_90_days"]
display(eda[numeric].describe().T)
display(eda[numeric].quantile([.25,.5,.75]).T.rename(columns={.25:"Q1",.5:"Median",.75:"Q3"}))
for col in ["annual_income", "total_spend_90_days"]:
    print(f"{col}: variance={eda[col].var():.2f}, std={eda[col].std():.2f}, skew={eda[col].skew():.2f}, kurtosis={eda[col].kurtosis():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(eda["age"], kde=True, ax=axes[0])
axes[0].set_title("Customer Age")
sns.histplot(eda["annual_income"], kde=True, ax=axes[1])
axes[1].set_title("Annual Income")
sns.boxplot(x=eda["total_spend_90_days"], ax=axes[2])
axes[2].set_title("90-Day Spend")
plt.tight_layout()
plt.show()

## 6. Categorical analysis and customer segments

In [ ]:
display(eda["membership"].value_counts().rename_axis("membership").to_frame("customers"))
display(eda["city"].value_counts().rename_axis("city").to_frame("customers"))
segment_summary = eda.groupby("membership", as_index=False).agg(
    customers=("customer_id", "nunique"),
    avg_spend=("total_spend_90_days", "mean"),
    median_spend=("total_spend_90_days", "median"),
    avg_orders=("orders_last_90_days", "mean"),
    avg_visits=("website_visits_month", "mean")
).sort_values("avg_spend", ascending=False)
display(segment_summary)
sns.barplot(data=segment_summary, x="membership", y="avg_spend")
plt.title("Average Spend by Membership")
plt.ylabel("Average 90-Day Spend")
plt.show()

## 7. Correlation analysis

In [ ]:
corr = eda[numeric].corr()
display(corr)
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Customer Metrics Correlation")
plt.show()
sns.scatterplot(data=eda, x="website_visits_month", y="total_spend_90_days", hue="membership", alpha=.75)
plt.title("Visits vs 90-Day Spend")
plt.show()

## 8. Outlier detection with IQR

In [ ]:
def iqr_outlier_table(data, column):
    q1, q3 = data[column].quantile([.25, .75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    flagged = data.loc[~data[column].between(lower, upper), ["customer_id", column]]
    return lower, upper, flagged

for col in ["annual_income", "total_spend_90_days", "website_visits_month"]:
    lower, upper, flagged = iqr_outlier_table(eda, col)
    print(f"{col}: IQR bounds {lower:.2f} to {upper:.2f}; flagged {len(flagged)}")
    display(flagged.head(10))

## 9. Text and date overview

In [ ]:
eda["last_purchase_date"] = pd.to_datetime(eda["last_purchase_date"], errors="coerce")
reference_date = eda["last_purchase_date"].max()
eda["days_since_last_purchase"] = (reference_date - eda["last_purchase_date"]).dt.days
print("Feedback categories:")
display(eda["feedback"].value_counts().to_frame("count"))
print("Average days since last purchase:", round(eda["days_since_last_purchase"].mean(), 1))
sns.histplot(eda["days_since_last_purchase"], bins=20)
plt.title("Days Since Last Purchase")
plt.show()

## 10. Basic statistical test

In [ ]:
from scipy.stats import f_oneway
groups = [group["total_spend_90_days"].dropna().to_numpy()
          for _, group in eda.groupby("membership")
          if len(group["total_spend_90_days"].dropna()) > 1]
if len(groups) >= 2:
    statistic, p_value = f_oneway(*groups)
    print(f"One-way ANOVA across membership groups: F={statistic:.3f}, p-value={p_value:.4f}")
    print("At alpha=0.05:", "at least one group mean may differ" if p_value < .05 else "insufficient evidence that group means differ")
    print("ANOVA does not tell which groups differ; post-hoc tests would be needed.")

## 11. Save outputs

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
eda.to_csv(output_dir / "customer_eda_cleaned.csv", index=False)
segment_summary.to_csv(output_dir / "membership_segment_summary.csv", index=False)
plt.figure(figsize=(8, 4))
sns.barplot(data=segment_summary, x="membership", y="avg_spend")
plt.title("Average Spend by Membership")
plt.tight_layout()
plt.savefig(output_dir / "average_spend_by_membership.png", dpi=150)
plt.close()
print("Saved outputs to:", output_dir.resolve())

## Conclusion

Describe customer profile, membership segment differences, spending and visit relationships, missing data handling, unusual values, and limitations. Synthetic data is for practice only; do not treat it as real customer evidence.